# DyNeRF reference: calibrated video to a held-out render

**Prepared, not yet GPU-validated.** Choose **Runtime > Change runtime type > T4 GPU**, then run cells in order. Stop at the first failure and retain the logs. No Drive connection is required. GPU availability/session length varies; see the [Colab FAQ](https://research.google.com/colaboratory/faq.html).

**English:** First verify execution (2 coarse + 10 fine steps). This does not establish reconstruction quality. The code is pinned; the proposed compatibility stack and sparse initialization differ from the published setup.

**Español:** Primero verificar ejecución (2 pasos coarse + 10 fine). Esto no demuestra calidad de reconstrucción. Entorno de compatibilidad propuesto e inicialización dispersa diferentes del artículo.

[Protocol and CPU evidence / Protocolo y evidencia CPU](https://github.com/craptou/Projet_4D_Gaussian_Splatting/blob/main/docs/progress/dynerf.en.md).

Before running, explain: what is a camera pose? Why must cam00 remain outside optimization? Why can a low reprojection error coexist with a poor image reconstruction?

## 1. Workspace and provenance / Espacio de trabajo y procedencia
Only writes to this Colab session. The Git revision and hardware are recorded in the run log. The repository is public; no token is needed.

In [ ]:
import os, sys, json, shutil, subprocess, urllib.request, hashlib, tarfile
from pathlib import Path
assert Path('/content').exists(), 'Run this notebook in Colab.'
ROOT = Path('/content/skate-trick-4dgs')
if not ROOT.exists():
    subprocess.run(['git','clone','--recurse-submodules','https://github.com/craptou/Projet_4D_Gaussian_Splatting.git',str(ROOT)],check=True)
UPSTREAM = ROOT / 'third_party/4DGaussians'
LOGS = ROOT / 'outputs/dynerf-colab-logs'
LOGS.mkdir(parents=True, exist_ok=True)
# All subprocess failures stop the cell; output is saved and also shown live.
def run(command, name, cwd=None):
    with (LOGS / (name + '.log')).open('w') as log:
        log.write('COMMAND: ' + repr([str(x) for x in command]) + '\n')
        log.flush()
        process = subprocess.Popen([str(x) for x in command], cwd=cwd, env=os.environ.copy(),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
        status = process.wait()
    if status:
        raise RuntimeError(f'{name} failed (exit {status}); see {LOGS / (name + ".log")}')
run(['nvidia-smi'], 'gpu')
revision = subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
upstream_revision = subprocess.check_output(['git','rev-parse','HEAD'],cwd=UPSTREAM,text=True).strip()
assert upstream_revision == '843d5ac636c37e4b611242287754f3d4ed150144'
(LOGS/'provenance.json').write_text(json.dumps(dict(project_revision=revision,upstream_revision=upstream_revision,notebook_status='started'),indent=2))
assert shutil.disk_usage('/content').free > 25 * 2**30, 'Need at least 25 GiB free before environment/data setup.'
print('Project revision:', revision)

## 2. Isolated environment / Entorno aislado
Colab changes its Python/PyTorch versions. Use a separate Python 3.10 interpreter and CUDA 12.1 toolkit. This is a proposed port, **not the original Python 3.7 / PyTorch 1.13.1 / CUDA 11.6 environment**. Solver/build success must be observed here. The toolkit and wheel downloads can take several minutes.

[PyTorch 2.1.2 official wheels](https://pytorch.org/get-started/previous-versions/#v212), [micromamba installation](https://mamba.readthedocs.io/en/latest/installation/micromamba-installation.html).

In [ ]:
MAMBA = Path('/content/micromamba/bin/micromamba')
ENV = Path('/content/skate4d-env')
os.environ['MAMBA_ROOT_PREFIX'] = '/content/skate4d-mamba'
if not MAMBA.exists():
    archive = Path('/content/micromamba.tar.bz2')
    urllib.request.urlretrieve('https://micro.mamba.pm/api/micromamba/linux-64/2.0.5', archive)
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == 'ff1a10b46e9ee673bb672ac464d4787b4e9c3569b75816a56f76dddc0b1f07fc', 'Installer checksum differs from inspected package'
    with tarfile.open(archive) as bundle:
        member = bundle.getmember('bin/micromamba')
        bundle.extract(member, '/content/micromamba', filter='data')
run([MAMBA,'create','--yes','--prefix',ENV,'--file',ROOT/'environment/environment-colab.yml'], 'environment')
PYTHON = ENV / 'bin/python'
os.environ['PATH'] = str(ENV/'bin') + os.pathsep + os.environ['PATH']
os.environ['CUDA_HOME'] = str(ENV)
os.environ['CONDA_PREFIX'] = str(ENV)
os.environ['CC'] = str(ENV/'bin/x86_64-conda-linux-gnu-gcc')
os.environ['CXX'] = str(ENV/'bin/x86_64-conda-linux-gnu-g++')
os.environ['CUDAHOSTCXX'] = os.environ['CXX']
os.environ['LD_LIBRARY_PATH'] = str(ENV/'lib') + ':' + str(ENV/'lib64') + ':' + os.environ.get('LD_LIBRARY_PATH','')
os.environ['MAX_JOBS'] = '2'
run([PYTHON,'-m','pip','install','torch==2.1.2','torchvision==0.16.2','--index-url','https://download.pytorch.org/whl/cu121'], 'pytorch-install')
run([PYTHON,'-m','pip','install','numpy==1.26.4','Pillow==9.5.0','opencv-python-headless==4.9.0.80',
     'yapf==0.40.1','scipy==1.11.4','matplotlib==3.7.5','lpips==0.1.4','plyfile==1.0.3',
     'pytorch-msssim==1.0.0','open3d==0.18.0','imageio==2.31.2','imageio-ffmpeg==0.4.9',
     'scikit-image==0.21.0','tqdm==4.66.1','tensorboard==2.15.2'], 'helpers-install')
# Install headless OpenCV first: MMCV detects it and avoids an extra GUI OpenCV wheel.
run([PYTHON,'-m','pip','install','mmcv==1.6.0'], 'mmcv-install')
run([PYTHON,'-m','pip','check'], 'pip-check')

## 3. Compile and verify CUDA / Compilar y comprobar CUDA
A driver reporting a CUDA version is not proof of a working compiler or extension. This cell must import both built modules. CUDA 12.1 does not support Blackwell compilation; select T4 (or another supported GPU).

In [ ]:
probe = """
import torch,json,sys
from pathlib import Path
assert torch.cuda.is_available(), 'No CUDA GPU available'
assert torch.version.cuda == '12.1', torch.version.cuda
cap=torch.cuda.get_device_capability(0)
assert cap <= (9,0), 'Use a T4/A100/L4/H100-compatible session, not a newer architecture'
Path(sys.argv[1]).write_text(json.dumps(dict(torch=torch.__version__,cuda=torch.version.cuda,architecture=f'{cap[0]}.{cap[1]}',gpu=torch.cuda.get_device_name(0))))
"""
run([PYTHON,'-c',probe,LOGS/'torch.json'], 'torch-probe')
os.environ['TORCH_CUDA_ARCH_LIST'] = json.loads((LOGS/'torch.json').read_text())['architecture']
run([ENV/'bin/nvcc','--version'], 'nvcc')
assert (ENV/'include/cuda_runtime.h').exists(), 'Toolkit headers missing from CUDA_HOME/include'
for name in ['depth-diff-gaussian-rasterization','simple-knn']:
    run([PYTHON,'-m','pip','install','--no-build-isolation','--no-deps',UPSTREAM/'submodules'/name], 'build-'+name)
run([PYTHON,'-c','import torch, diff_gaussian_rasterization, simple_knn._C; assert torch.cuda.is_available(); print("CUDA extension imports passed")'], 'extensions')
run([PYTHON,'-m','pip','freeze'], 'pip-freeze')
run([MAMBA,'list','--prefix',ENV,'--explicit'], 'conda-explicit')

## 4. Official data and sparse initialization / Datos oficiales e inicialización
Archive: ~1.14 GB; extracted videos and frames need more storage. Data: Neural 3D Video Synthesis, Li et al., CVPR 2022, **CC-BY-NC-4.0**. [Official release](https://github.com/facebookresearch/Neural_3D_Video/releases/tag/v1.0).

The compact NPZ was triangulated locally from frame 0 of 19 training cameras, keeping supplied calibration fixed. It is a sparse substitute for upstream dense stereo. cam00 colors did not enter that map. Supplied full-rig calibration remains in use.

In [ ]:
if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    run(['apt-get','update','-qq'], 'apt-update')
    run(['apt-get','install','--yes','ffmpeg'], 'ffmpeg-install')
DATA = ROOT / 'data/sample/dynerf'
DATA.mkdir(parents=True, exist_ok=True)
ARCHIVE = DATA / 'cut_roasted_beef.zip'
URL = 'https://github.com/facebookresearch/Neural_3D_Video/releases/download/v1.0/cut_roasted_beef.zip'
if not ARCHIVE.exists():
    run(['curl','--fail','--location','--output',ARCHIVE,URL], 'data-download')
with ARCHIVE.open('rb') as handle:
    actual_hash = hashlib.file_digest(handle,'sha256').hexdigest()
assert actual_hash == '2f9d3bd8782b96425e8adb506fa4cbdfaab4ab753a8fdaf44b9b2468000b8c4f', 'Archive hash differs from local reference'
SCENE = DATA / 'cut_roasted_beef'
run([PYTHON,ROOT/'scripts/inspect_dynerf.py','--archive',ARCHIVE,'--scene',SCENE,'--output',LOGS/'inspection'], 'data-inspection', ROOT)
restore = """
import numpy as np,sys
from plyfile import PlyData,PlyElement
p=np.load(sys.argv[1],allow_pickle=False)
xyz,rgb=p['xyz'],p['rgb']
assert xyz.shape == rgb.shape == (3390,3)
v=np.zeros(len(xyz),dtype=[('x','f4'),('y','f4'),('z','f4'),('nx','f4'),('ny','f4'),('nz','f4'),('red','u1'),('green','u1'),('blue','u1')])
for i,n in enumerate(['x','y','z']): v[n]=xyz[:,i]
for i,n in enumerate(['red','green','blue']): v[n]=rgb[:,i]
PlyData([PlyElement.describe(v,'vertex')]).write(sys.argv[2])
"""
run([PYTHON,'-c',restore,ROOT/'docs/evidence/dynerf-reference-init.npz',SCENE/'points3D_downsample2.ply'], 'initialization')

## 5. Loader gate / Comprobar el cargador
Pinned loader: 300 frames per camera, 1352 x 1014 pixels, time = index / 300. It can take several minutes to extract all images. cam04 is absent: retain sorted existing filenames. Expect 19 x 300 training and 1 x 300 test samples.

In [ ]:
run([PYTHON,UPSTREAM/'scripts/preprocess_dynerf.py','--datadir',SCENE], 'frame-extraction', UPSTREAM)
loader_probe = """
import sys
from scene.neural_3D_dataset_NDC import Neural3D_NDC_Dataset
train=Neural3D_NDC_Dataset(sys.argv[1],'train',1.0,eval_index=0)
test=Neural3D_NDC_Dataset(sys.argv[1],'test',1.0,eval_index=0)
assert len(train)==5700 and len(test)==300
assert all('/cam00/' not in p for p in train.image_paths)
assert all('/cam00/' in p for p in test.image_paths)
a=train[0]; b=test[299]
assert tuple(a[0].shape)==(3,1014,1352)
assert abs(b[2]-299/300)<1e-7
print('Loader/split/shape/time passed:',len(train),len(test))
"""
run([PYTHON,'-c',loader_probe,SCENE], 'loader-check', UPSTREAM)

## 6. Bounded training / Entrenamiento limitado
**Execution only:** 2 coarse + 10 fine steps, batch size 1, no densification. It should produce a saved model, not a converged reconstruction. No long training job starts automatically. Keep cam00 and settings fixed for future quality evaluation.

In [ ]:
MODEL = ROOT/'outputs/dynerf-colab-smoke'
CONFIG = ROOT/'configs/experiments/dynerf-smoke.py'
assert not MODEL.exists(), 'Use a fresh model path; do not overwrite a previous experiment.'
run([PYTHON,'train.py','-s',SCENE,'--model_path',MODEL,'--expname','dynerf-colab-smoke',
     '--configs',CONFIG,'--save_iterations','10','--test_iterations','10','--ip','127.0.0.1'], 'training', UPSTREAM)
for name in ['point_cloud.ply','deformation.pth']:
    artifact=MODEL/'point_cloud/iteration_10'/name
    assert artifact.exists() and artifact.stat().st_size>0, str(artifact)
print('Saved-model gate passed; quality is still untested.')

## 7. Three held-out renders / Tres renders reservados
Use the **supplied cam00 pose**, not the diagnostic PnP estimate. Render only source frames 0, 149, 299 to keep output bounded. The output filenames 00000/00001/00002 map to these source indices. These are sparse visual checks, not a full benchmark.

In [ ]:
render_probe = """
import sys,json,torch
from pathlib import Path
from argparse import ArgumentParser
import mmcv
from arguments import ModelParams,ModelHiddenParams,PipelineParams
from utils.params_utils import merge_hparams
from scene import Scene,GaussianModel
from render import render_set
p=ArgumentParser()
lp=ModelParams(p); hp=ModelHiddenParams(p); pp=PipelineParams(p)
a=p.parse_args(['-s',sys.argv[1],'-m',sys.argv[2]])
a=merge_hparams(a,mmcv.Config.fromfile(sys.argv[3]))
with torch.no_grad():
    dataset=lp.extract(a)
    gaussians=GaussianModel(dataset.sh_degree,hp.extract(a))
    scene=Scene(dataset,gaussians,load_iteration=10,shuffle=False)
    indices=[0,149,299]
    views=[scene.getTestCameras()[i] for i in indices]
    background=torch.tensor([1,1,1] if dataset.white_background else [0,0,0],dtype=torch.float32,device='cuda')
    render_set(dataset.model_path,'test',10,views,gaussians,pp.extract(a),background,scene.dataset_type)
Path(sys.argv[2],'render-indices.json').write_text(json.dumps(dict(camera='cam00',source_frames=indices,times=[i/300 for i in indices],status='12-step execution check; not reconstruction-quality evaluation'),indent=2))
"""
run([PYTHON,'-c',render_probe,SCENE,MODEL,CONFIG], 'rendering', UPSTREAM)
from IPython.display import display, Image as NotebookImage
for index in range(3):
    for kind in ['gt','renders']:
        path=MODEL/'test/ours_10'/kind/f'{index:05d}.png'
        assert path.exists() and path.stat().st_size>0, str(path)
        print(kind, 'source frame', [0,149,299][index])
        display(NotebookImage(filename=str(path),width=500))
(LOGS/'success.json').write_text(json.dumps(dict(status='GPU execution gates passed',quality_evaluated=False,source_frames=[0,149,299]),indent=2))

## 8. Save evidence before disconnecting / Guardar evidencias
Colab session files are temporary. Download logs and model/render artifacts. The dataset itself is excluded from this export. If an earlier cell failed, this cell can still export the logs collected so far.

In [ ]:
export = Path('/content/dynerf-reference-evidence.zip')
import zipfile
with zipfile.ZipFile(export,'w',zipfile.ZIP_DEFLATED) as bundle:
    for base in [LOGS, ROOT/'outputs/dynerf-colab-smoke']:
        if base.exists():
            for path in base.rglob('*'):
                if path.is_file():
                    bundle.write(path,str(path.relative_to(ROOT/'outputs')))
from google.colab import files
files.download(str(export))

## Interpret the result / Interpretar el resultado

- A failure is evidence about that gate; do not skip it or call later stages successful.
- Successful renders after 12 optimization steps confirm execution, not useful dynamic reconstruction.
- Next: explain the stages, review render/target pairs, then freeze a longer-run protocol and evaluate held-out PSNR/SSIM/LPIPS. Keep the sparse-initialization and compatibility-stack departures explicit.
- Return to True Skate only after this gate, first checking the fixed-camera mode and then per-frame pose loading if using moving cameras.